# Psychometric fits (labdata)

Uses fitted `PsychometricSubjectFit` rows from a seeded behavior analysis set.
For batch figures without a notebook, prefer:

```bash
uv run python scripts/analyses/plot_psychometrics.py --analysis-set-id <id> --output figures/psychometrics.pdf
```

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

REPO_ROOT = (
    Path.cwd().parent if Path.cwd().name == "psychometric_curves" else Path.cwd()
)
for path in [REPO_ROOT, REPO_ROOT / "src"]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from behavior_analyses.psychometrics import cumulative_gaussian
from labdata_plugin.analysisschema import PsychometricSubjectFit

ANALYSIS_SET_ID = "example_analysis_set"  # replace after seeding
rows = (
    PsychometricSubjectFit() & {"analysis_set_id": ANALYSIS_SET_ID, "fit_status": "fit"}
).fetch(as_dict=True)
assert rows, f"No fitted psychometrics for {ANALYSIS_SET_ID}"

fig, ax = plt.subplots(figsize=(5, 5))
for row in rows:
    stims = np.asarray(row["stims"], dtype=float)
    params = np.asarray(
        [row["bias"], row["sensitivity"], row["guess_rate"], row["lapse_rate"]]
    )
    p_right = np.asarray(row["p_right"], dtype=float)
    x = np.asarray(sorted(stims), dtype=float)
    label = f"{row['subject_name']} (n={row['n_choices_fit']})"
    ax.plot(x, cumulative_gaussian(*params, x), label=label)
    ax.plot(stims, p_right, "o", ms=4)
ax.set_xlabel("Stimulus rate relative to boundary (Hz)")
ax.set_ylabel("P(right choice)")
ax.set_ylim(0, 1)
ax.legend(frameon=False, fontsize=8)
ax.set_title("Psychometric fits by subject")
fig.show()